## Filter domainome and create subset files for MaSIF search

Filter domainome based on pre-computed metrics and write subset text files for MaSIF search 

In [ ]:
from pathlib import Path
from functools import partial
import pandas as pd
import numpy as np
from tqdm import tqdm
tqdm.pandas()

import os
import sys

root_dir = !git rev-parse --show-toplevel
root_dir = root_dir[0]
os.chdir(root_dir)

In [ ]:
# Define absolute paths

# Path to domainome root directory
domainome_root = os.path.abspath(os.path.normpath("./DPAM-AI_AFDB_domainome_v6"))

# Path to domainome metrics .csv file
domainome_metrics_csv = os.path.join(domainome_root, "DPAM_AI_AFDB_domainome_v6_domains_info_legacy.csv")

# Path to domainome .pdb directory
domainome_pdb_dir = os.path.join(domainome_root, "pdbs")

# Path to output directory for filtered domainome
filtered_domainome_dir = os.path.join(domainome_root, "folded_intracellular_domainome_260310")

___
### Interactively inspect domainome

In [ ]:
# ---------- Define helper functions ----------

from Bio.PDB import PDBParser
import py3Dmol

parser = PDBParser()

# Function to return the view object for a PDB file
def view_pdb(pdb_path):
    """
    Visualize a PDB structure as a cartoon in py3Dmol and return the view.

    Args:
        pdb_path (str): Path to the PDB file.

    Returns:
        py3Dmol.view: The py3Dmol view object displaying the structure.
    """
    view = py3Dmol.view(width=500, height=500)
    with open(pdb_path, 'r') as f:
        pdb_str = f.read()
    view.addModel(pdb_str, "pdb")
    view.setStyle({"model": 0}, {"cartoon": {"color": "lightgrey"}})
    view.zoomTo()
    return view

In [ ]:
# Read domainome metrics .csv file
df_domainome_metrics = pd.read_csv(domainome_metrics_csv)

In [ ]:
df_domainome_metrics.iloc[0]

In [ ]:
# ----- Interactive scatter plot with py3Dmol view rendering on click -----
import plotly.express as px
import plotly.graph_objs as go
from IPython.display import display, clear_output
import ipywidgets as widgets

def render_plotly_interactive(df, x_axis, y_axis, color_by):
    fig = px.scatter(
        df,
        x=x_axis,
        y=y_axis,
        color=color_by,
        hover_data=["id"],
        color_continuous_scale="viridis",
        width=800,
        height=600
    )
    # Set customdata PER TRACE so indices align with each trace's points
    for trace in fig.data:
        if len(fig.data) == 1:
            # Continuous color: single trace with all points
            subset = df
        else:
            # Categorical color: each trace has a subset
            mask = df[color_by] == trace.name
            subset = df[mask]
        trace.customdata = np.column_stack([
            subset.index.values,
            subset["id"].values
        ])
    fig.update_traces(
        hovertemplate="%{x}<br>%{y}<br>id: %{customdata[1]}<extra></extra>"
    )
    figw = go.FigureWidget(fig)
    
    viewer_output = widgets.Output()
    row_output = widgets.Output()

    def handle_click(trace, points, selector):
        if len(points.point_inds) > 0:
            idx = points.point_inds[0]
            row_idx = trace.customdata[idx][0]

            clear_output(wait=True)
            display(figw, viewer_output, row_output)

            row = df.loc[row_idx]
            pdb_path = os.path.join(domainome_pdb_dir, f"{row['id']}.pdb")

            # Render the corresponding py3Dmol view ABOVE the row info
            with viewer_output:
                viewer_output.clear_output(wait=True)
                view = view_pdb(pdb_path)
                display(view)

            # Prepare the column list: first 10 columns plus x_axis, y_axis, color_by (add any not already in first 10)
            col_first_10 = list(df.columns[:10])
            special_cols = [axis for axis in [x_axis, y_axis, color_by] if axis not in col_first_10]
            col_selection = col_first_10 + special_cols

            # Show row info BELOW the py3Dmol viewer -- only desired columns
            with row_output:
                row_output.clear_output(wait=True)
                display(row[col_selection])

    for trace in figw.data:
        trace.on_click(handle_click)

    display(figw, viewer_output, row_output)

# Example call:
x_axis = "norm_sasa"
y_axis = "DPAM_prob"
color_by = "Judge"

# Ensure x_axis and y_axis are numeric
df_domainome_metrics[x_axis] = pd.to_numeric(df_domainome_metrics[x_axis], errors='coerce')
df_domainome_metrics[y_axis] = pd.to_numeric(df_domainome_metrics[y_axis], errors='coerce')

render_plotly_interactive(
    df_domainome_metrics,
    x_axis,
    y_axis,
    color_by
)

Filtering strategies:
- Remove all entries whose total_n_sse < 4
- Remove all entries whose intracellular_fraction <0.5
- Remove all entries whose norm_SASA > 95
- For simple_topology:
    - Remove entries whose DPAM_prob < 0.8

In [ ]:
# Filtering strategies:
filters = {
    "total_n_sse": {"min": 4},
    "intracellular_frac": {"min": 0.5},
    "norm_sasa": {"max": 95}
}

# Function to apply filters to the domainome metrics dataframe
def apply_filters(df, filters):
    df_filtered = df.copy()

    # Ensure all columns are numeric
    df_filtered[list(filters.keys())] = df_filtered[list(filters.keys())].apply(pd.to_numeric, errors='coerce') 

    for col, rule in filters.items():
        if "min" in rule:
            df_filtered = df_filtered[df_filtered[col] >= rule["min"]]
        if "max" in rule:
            df_filtered = df_filtered[df_filtered[col] <= rule["max"]]
    return df_filtered.reset_index(drop=True)

In [ ]:
# Apply the filters
df_filtered = apply_filters(df_domainome_metrics, filters)

# For entries whose Judge is "simple_topology", remove entries whose DPAM_prob < 0.8
df_filtered["DPAM_prob"] = pd.to_numeric(df_filtered["DPAM_prob"], errors='coerce')
mask = ~((df_filtered["Judge"] == "simple_topology") & (df_filtered["DPAM_prob"] < 0.8))
df_filtered = df_filtered[mask]

print(f"Shape before filtering: {df_domainome_metrics.shape}")
print(f"Shape after filtering: {df_filtered.shape}")


In [ ]:
# Visualize the filtered domainome
x_axis = "norm_sasa"
y_axis = "DPAM_prob"
color_by = "Judge"

# Ensure x_axis and y_axis are numeric
df_filtered[x_axis] = pd.to_numeric(df_filtered[x_axis], errors='coerce')
df_filtered[y_axis] = pd.to_numeric(df_filtered[y_axis], errors='coerce')

render_plotly_interactive(
    df_filtered,
    x_axis,
    y_axis,
    color_by
)

In [ ]:
# Save df_filtered to a new csv file
os.makedirs(filtered_domainome_dir, exist_ok=True)
df_filtered.to_csv(os.path.join(filtered_domainome_dir, "filtered_AFDB_domainome_v6_metrics_260310.csv"), index=False)


In [ ]:
# Create subset text files
all_ids = df_filtered["id"].tolist()

# Create a text file for each group of 50 ids
subset_dir = os.path.join(filtered_domainome_dir, "subsets")
os.makedirs(subset_dir, exist_ok=True)

# Determine the number of groups
num_groups = len(all_ids) // 50

for i in range(0, num_groups+1):
    group_ids = all_ids[i*50:(i+1)*50]
    file_path = os.path.join(subset_dir, f"{i+1}")
    with open(file_path, "w") as f:
        for id in group_ids:
            f.write(f"{id}\n")
